In [54]:
import os
from dotenv import load_dotenv
import requests
from datetime import datetime
import uuid
import io
import zipfile
import pandas as pd
from dateutil.relativedelta import relativedelta
from pathlib import Path
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side


In [2]:
load_dotenv()
TOKEN_KEY = os.getenv('КОСТРИК')
HEADERS = {'Authorization': TOKEN_KEY}

In [3]:
url = 'https://seller-analytics-api.wildberries.ru/api/v2/nm-report/downloads'
date = '01.03.2026'
date =  str(datetime.strptime(date, "%d.%m.%Y").strftime("%Y-%m-%d"))
new_uuid = str(uuid.uuid4())
reportType = 'DETAIL_HISTORY_REPORT'
params = {
    'startDate': date,
    'endDate': date,
    'skipDeletedNm': False
}

body = {
    'id': new_uuid,
    'reportType': reportType,
    'params': params
}

In [4]:
response = requests.post(url=url, headers=HEADERS, json=body)
print(response.status_code)
print(response.json())

200
{'data': 'Началось формирование файла/отчета'}


In [83]:
url = 'https://seller-analytics-api.wildberries.ru/api/v2/nm-report/downloads'
body = {
    # 'filter[downloadIds]': new_uuid
}

In [81]:
body

{'filter[downloadIds]': 'ce9eb79f-09e0-41e2-aaba-918da3ce77aa'}

In [84]:
response = requests.get(url=url, headers=HEADERS, params=body)
print(response.status_code)
print(response.json())

200
{'data': [{'id': 'a85a57cd-21bb-4148-b3f6-7a8a6bf18c92', 'status': 'SUCCESS', 'name': 'detail_history_report', 'size': 2079, 'startDate': '2026-03-01', 'endDate': '2026-03-01', 'createdAt': '2026-03-19 11:20:00'}, {'id': '4e3f152a-eb55-4440-a34f-7b36c6371808', 'status': 'SUCCESS', 'name': 'detail_history_report', 'size': 2079, 'startDate': '2026-03-01', 'endDate': '2026-03-01', 'createdAt': '2026-03-19 11:19:33'}, {'id': '121b8f59-f94c-4bd3-8d24-eb1341d0a189', 'status': 'SUCCESS', 'name': 'detail_history_report', 'size': 2079, 'startDate': '2026-03-01', 'endDate': '2026-03-01', 'createdAt': '2026-03-19 11:16:34'}, {'id': '96914cf6-109e-4a8a-8062-6473e61af89a', 'status': 'SUCCESS', 'name': 'detail_history_report', 'size': 2079, 'startDate': '2026-03-01', 'endDate': '2026-03-01', 'createdAt': '2026-03-19 11:14:56'}, {'id': '7472b4b7-e6f7-41bf-b76f-ea8c18e0f50b', 'status': 'SUCCESS', 'name': 'detail_history_report', 'size': 2079, 'startDate': '2026-03-01', 'endDate': '2026-03-01', 'cr

In [11]:
url = f'https://seller-analytics-api.wildberries.ru/api/v2/nm-report/downloads/file/{new_uuid}'

In [13]:
response = requests.get(url=url, headers=HEADERS)
print(response.status_code)

200


In [47]:
if response.status_code == 200:
        zip_buffer = io.BytesIO(response.content)
        
        with zipfile.ZipFile(zip_buffer, 'r') as zip_file:
            file_content = pd.read_csv(io.StringIO(zip_file.read(f'{new_uuid}.csv').decode('utf-8')), index_col=None)
            date = file_content['dt'].iloc[0]
            ordersCount = file_content['ordersCount'].sum() - file_content['cancelCount'].sum()   
            ordersSum = file_content['ordersSumRub'].sum() - file_content['cancelSumRub'].sum()  
            buyoutsCount = file_content['buyoutsCount'].sum()
            buyoutsSum = file_content['buyoutsSumRub'].sum()
            openCard = file_content['openCardCount'].sum()
            addToCart = file_content['addToCartCount'].sum()
            addToCartConversion = file_content.loc[file_content['addToCartConversion'] > 0, 'addToCartConversion'].mean()
            cartToOrderConversion = file_content.loc[file_content['cartToOrderConversion'] > 0, 'cartToOrderConversion'].mean()
            buyoutPercent = file_content.loc[file_content['buyoutPercent'] > 0, 'buyoutPercent'].mean()
            addToWishlist = file_content['addToWishlist'].sum()
            
            day_stats = [{
                'Обновлено': (datetime.now() - relativedelta(hours=3)).strftime('%d.%m.%Y %H:%M'),
                'Дата': datetime.strptime(date, '%Y-%m-%d').strftime('%d.%m.%Y'),
                'Заказано, шт.': ordersCount,
                'Заказано, руб.': ordersSum,
                'Выкуплено, шт.': buyoutsCount,
                'Выкуплено, руб.': buyoutsSum,
                'Переходов в карточки': openCard,
                'Добавлений в корзину': addToCart,
                'Средняя конверсия в корзину': addToCartConversion,
                'Средняя конверсия в заказ': cartToOrderConversion,
                'Средний процент выкупа': buyoutPercent,
                'Добавлений в Отложенные': addToWishlist
            }]
            
            

In [63]:

filename = 'Ежедневная статистика.xlsx'
df_new = pd.DataFrame(day_stats)
file_path = Path(filename)

try:
    if file_path.exists():
        df_existing = pd.read_excel(file_path)
        df_combined = pd.concat([df_existing, df_new], ignore_index=True)
    else:
        df_combined = df_new
    df_combined.to_excel(file_path, index=False, engine='openpyxl')

    wb = load_workbook(filename)
    ws = wb.active
        
    header_font = Font(bold=True, color='000000')
    header_fill = PatternFill(start_color='CCECFF', fill_type='solid')
    thin_border = Border(
            left=Side(style='thin'),
            right=Side(style='thin'),
            top=Side(style='thin'),
            bottom=Side(style='thin')
        )
        
    for cell in ws[1]:
        cell.font = header_font
        cell.fill = header_fill
        cell.border = thin_border

    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
        for cell in row:
            cell.border = thin_border

    for col in ws.columns:
        max_length = 0
        column_letter = col[0].column_letter
        
        for cell in col:
            try:
                if len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass
        
        adjusted_width = min(max_length + 5, 60)
        ws.column_dimensions[column_letter].width = adjusted_width

    wb.save(filename)
    wb.close()

    print(f"Данные сохранены в {filename}")
    
except PermissionError:
    print('Нет доступа!')

Данные сохранены в Ежедневная статистика.xlsx


In [64]:
# Создаём диапазон дат
dates_list = (pd.date_range(end=pd.Timestamp.now(), periods=30, freq='D')).strftime('%Y-%m-%d').tolist()

['2026-02-18',
 '2026-02-19',
 '2026-02-20',
 '2026-02-21',
 '2026-02-22',
 '2026-02-23',
 '2026-02-24',
 '2026-02-25',
 '2026-02-26',
 '2026-02-27',
 '2026-02-28',
 '2026-03-01',
 '2026-03-02',
 '2026-03-03',
 '2026-03-04',
 '2026-03-05',
 '2026-03-06',
 '2026-03-07',
 '2026-03-08',
 '2026-03-09',
 '2026-03-10',
 '2026-03-11',
 '2026-03-12',
 '2026-03-13',
 '2026-03-14',
 '2026-03-15',
 '2026-03-16',
 '2026-03-17',
 '2026-03-18',
 '2026-03-19']